# BERT Binary Classification 모델 구현 실습 - PyTorch

이 노트북은 영화 리뷰 데이터를 이용한 **BERT 기반 이진 감성 분류 모델**을 PyTorch 코드로 구현합니다.

## PDF 내용 요약

- 영화평 데이터 세트를 Pandas로 읽고 `id`, `review`, `sentiment` 컬럼을 구성합니다.
- `train_test_split`과 `stratify`를 사용하여 Train/Test, Train/Validation 데이터를 분리합니다.
- `torch.utils.data.Dataset`을 상속받아 BERT 입력 형식인 `input_ids`, `attention_mask`, `labels`를 반환하는 Dataset 클래스를 정의합니다.
- Train/Validation 데이터로 Dataset 객체를 생성합니다.
- Hugging Face에서 한국어 BERT 모델인 `kykim/bert-kor-base`를 다운로드합니다.
- Fine-tuning 전략을 설정하여 일부 BERT 계층을 고정하거나 마지막 계층만 학습하도록 구성합니다.
- Epoch, Batch Size, Learning Rate, Weight Decay 같은 학습 하이퍼파라미터를 설정합니다.
- Test Dataset을 생성하고 평가 준비를 합니다.
- 학습된 모델로 Test 데이터를 예측하고 정확도, 정밀도, 재현율, F1-score를 확인합니다.


## 1. 패키지 설치

 `transformers` 설치. 여기서는 PyTorch 기반 BERT 학습에 필요한 패키지를 함께 설치합니다.

In [2]:
# Colab 환경에서 필요한 패키지를 설치합니다.
# transformers: Hugging Face의 BERT 모델과 Tokenizer를 사용하기 위한 라이브러리입니다.
# accelerate: PyTorch 학습 장치 설정을 보조하는 라이브러리로, 최신 transformers와 함께 자주 사용됩니다.
# scikit-learn: 데이터 분리와 평가 지표 계산에 사용합니다.
!pip -q install transformers accelerate scikit-learn

## 2. 라이브러리 불러오기와 재현성 설정

실험 결과가 실행할 때마다 크게 달라지지 않도록 난수 시드를 고정합니다.

In [1]:
# 운영체제 경로 확인, 파일 존재 여부 확인 등에 사용하는 표준 라이브러리입니다.
import os

# 난수 시드를 고정하기 위해 사용하는 표준 라이브러리입니다.
import random

# 수치 계산과 배열 처리를 위해 NumPy를 불러옵니다.
import numpy as np

# 표 형태 데이터를 읽고 전처리하기 위해 Pandas를 불러옵니다.
import pandas as pd

# PyTorch의 핵심 기능인 Tensor, 모델, 학습 연산을 사용하기 위해 torch를 불러옵니다.
import torch

# Dataset은 사용자 정의 데이터셋 클래스를 만들 때 상속받고, DataLoader는 미니배치 단위로 데이터를 공급합니다.
from torch.utils.data import Dataset, DataLoader

# train_test_split은 Train, Validation, Test 데이터를 분리할 때 사용합니다.
from sklearn.model_selection import train_test_split

# classification_report는 분류 모델의 정밀도, 재현율, F1-score를 한 번에 출력합니다.
from sklearn.metrics import accuracy_score, classification_report

# BertTokenizerFast는 문장을 BERT 입력 숫자 토큰으로 변환합니다.
from transformers import BertTokenizerFast

# BertForSequenceClassification은 문장 분류용 출력층이 붙어 있는 BERT 모델입니다.
from transformers import BertForSequenceClassification

# AdamW는 Transformer 계열 모델에서 자주 사용하는 Adam 기반 최적화 함수입니다.
from torch.optim import AdamW

# get_linear_schedule_with_warmup은 학습률을 처음에는 서서히 올리고 이후 점차 낮추는 스케줄러입니다.
from transformers import get_linear_schedule_with_warmup

# tqdm은 반복문의 진행률을 시각적으로 보여줍니다.
from tqdm.auto import tqdm

# 실험 재현성을 위해 사용할 난수 시드 값을 지정합니다.
SEED = 42

# 파이썬 random 모듈의 난수 시드를 고정합니다.
random.seed(SEED)

# NumPy 난수 시드를 고정합니다.
np.random.seed(SEED)

# PyTorch CPU 난수 시드를 고정합니다.
torch.manual_seed(SEED)

# CUDA GPU가 사용 가능하다면 GPU 난수 시드도 고정합니다.
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# GPU가 있으면 cuda를 사용하고, 없으면 cpu를 사용하도록 장치를 설정합니다.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 현재 사용 중인 학습 장치를 출력합니다.
print("사용 장치:", device)

사용 장치: cuda


## 3. 영화 리뷰 데이터 로드

영화평 데이터 세트를 업로드하거나 Google Drive 경로에서 읽습니다. 데이터는 `id`, `review`, `sentiment` 컬럼으로 구성됩니다.

In [5]:
# Google Colab에서 실행 중인지 확인하기 위해 try 문을 사용합니다.
try:
    # Colab에서 Google Drive를 연결하기 위한 모듈입니다.
    from google.colab import drive

    # Google Drive를 /content/mnt 경로에 마운트합니다.
    drive.mount("/content/mnt")

# 로컬 환경이나 Colab이 아닌 환경에서는 google.colab 모듈이 없으므로 예외가 발생할 수 있습니다.
except Exception as e:
    # Colab 환경이 아니면 Drive 마운트를 건너뛰고 안내 메시지만 출력합니다.
    print("Google Drive 마운트를 건너뜁니다:", e)

# 기본 데이터 파일 경로를 지정합니다. 필요하면 본인 환경에 맞게 수정합니다.
DATA_PATH = "/content/mnt/MyDrive/data/ratings_train.txt"

# 지정한 경로에 파일이 없을 때 사용할 대체 경로를 지정합니다.
LOCAL_FALLBACK_PATH = "./ratings_train.txt"

# Google Drive 경로에 파일이 있으면 해당 경로를 사용합니다.
if os.path.exists(DATA_PATH):
    data_path = DATA_PATH

# 현재 작업 폴더에 ratings_train.txt가 있으면 대체 경로를 사용합니다.
elif os.path.exists(LOCAL_FALLBACK_PATH):
    data_path = LOCAL_FALLBACK_PATH

# 두 경로 모두 없으면 파일 업로드가 필요하다는 오류를 발생시킵니다.
else:
    raise FileNotFoundError("ratings_train.txt 파일을 Google Drive의 /MyDrive/data/ 또는 현재 폴더에 배치하세요.")

# 탭으로 구분된 영화 리뷰 파일을 Pandas DataFrame으로 읽습니다.
dataset = pd.read_table(data_path)

# PDF 예제와 동일하게 컬럼명을 id, review, sentiment로 지정합니다.
dataset.columns = ["id", "review", "sentiment"]

# 데이터가 정상적으로 읽혔는지 상위 5개 행을 확인합니다.
dataset.head()

Google Drive 마운트를 건너뜁니다: Error: credential propagation was unsuccessful


,id,review,sentiment
0,9976970,더빙이 실망스럽네요..,0
1,3819312,오버연기조차 가볍지 않구나,1
2,10265843,너무재밌었다. 그래서보는것을추천한다,0
3,9045019,솔직히 재미는 없다..평점 조정,0
4,6483659,배우의 익살스런 연기가 돋보였던 영화!,1


## 4. 결측치 제거와 레이블 분포 확인

영화 리뷰 문장이 비어 있으면 Tokenizer가 처리할 수 없으므로 결측치를 제거합니다. 레이블 분포를 확인하면 긍정/부정 데이터가 균형적인지 판단할 수 있습니다.

In [6]:
# 결측치 제거 전 데이터 개수를 저장합니다.
before_count = len(dataset)

# review 또는 sentiment에 결측치가 있는 행을 제거합니다.
dataset = dataset.dropna(subset=["review", "sentiment"]).reset_index(drop=True)

# sentiment 컬럼을 정수형으로 변환하여 PyTorch label로 사용할 수 있게 합니다.
dataset["sentiment"] = dataset["sentiment"].astype(int)

# 결측치 제거 후 데이터 개수를 저장합니다.
after_count = len(dataset)

# 제거된 행 개수를 출력합니다.
print("결측치 제거 전 데이터 수:", before_count)

# 남은 행 개수를 출력합니다.
print("결측치 제거 후 데이터 수:", after_count)

# sentiment 값별 데이터 개수를 출력합니다.
print(dataset["sentiment"].value_counts())

결측치 제거 전 데이터 수: 150000
결측치 제거 후 데이터 수: 149995
sentiment
0    75170
1    74825
Name: count, dtype: int64


## 5. Train / Validation / Test 데이터 분리

 `stratify`를 적용하여 레이블 비율을 유지하면서 데이터를 나눕니다. 먼저 전체 데이터를 Train 80%, Test 20%로 나누고, 다시 Train을 Train 70%, Validation 30%로 나눕니다.

In [7]:
# 전체 데이터의 인덱스와 레이블을 기준으로 Train/Test 인덱스를 분리합니다.
train_idx, test_idx, _, _ = train_test_split(
    dataset.index,                 # 분리할 전체 데이터의 인덱스입니다.
    dataset["sentiment"],          # stratify에 사용할 레이블입니다.
    test_size=0.2,                 # 전체 데이터 중 20%를 Test 데이터로 사용합니다.
    stratify=dataset["sentiment"], # 긍정/부정 비율이 유지되도록 층화 샘플링을 적용합니다.
    random_state=SEED              # 같은 결과가 나오도록 난수 시드를 고정합니다.
)

# Train 인덱스에 해당하는 행을 선택합니다.
train_set = dataset.iloc[train_idx].reset_index(drop=True)

# Test 인덱스에 해당하는 행을 선택합니다.
test_set = dataset.iloc[test_idx].reset_index(drop=True)

# Train 데이터에서 다시 Train/Validation 인덱스를 분리합니다.
train_idx, valid_idx, _, _ = train_test_split(
    train_set.index,                    # 다시 분리할 Train 데이터의 인덱스입니다.
    train_set["sentiment"],             # stratify에 사용할 Train 데이터의 레이블입니다.
    test_size=0.3,                      # 기존 Train 중 30%를 Validation으로 사용합니다.
    stratify=train_set["sentiment"],    # Train/Validation에도 레이블 비율을 유지합니다.
    random_state=SEED                   # 같은 결과가 나오도록 난수 시드를 고정합니다.
)

# Validation 인덱스에 해당하는 행을 선택합니다.
valid_set = train_set.iloc[valid_idx].reset_index(drop=True)

# 최종 Train 인덱스에 해당하는 행을 선택합니다.
train_set = train_set.iloc[train_idx].reset_index(drop=True)

# 각 데이터셋의 크기를 출력합니다.
print("Train:", train_set.shape)

# Validation 데이터셋의 크기를 출력합니다.
print("Validation:", valid_set.shape)

# Test 데이터셋의 크기를 출력합니다.
print("Test:", test_set.shape)

Train: (83997, 3)
Validation: (35999, 3)
Test: (29999, 3)


## 6. BERT Dataset 클래스 정의

 PyTorch의 `Dataset`을 상속하여 하나의 리뷰 문장을 BERT 입력 형식인 `input_ids`, `attention_mask`, `labels`로 변환합니다.

In [11]:
# 영화 리뷰 감성 분류용 Dataset 클래스를 정의합니다.
class BertMovieReviewDataset(Dataset):
    # Dataset 객체가 생성될 때 리뷰, 레이블, Tokenizer, 최대 길이를 저장합니다.
    def __init__(self, reviews, sentiments, tokenizer, max_len=128):
        # 리뷰 문장 리스트를 객체 변수에 저장합니다.
        self.reviews = reviews

        # 정답 레이블 리스트를 객체 변수에 저장합니다.
        self.sentiments = sentiments

        # BERT Tokenizer를 객체 변수에 저장합니다.
        self.tokenizer = tokenizer

        # 모든 문장의 최대 토큰 길이를 저장합니다.
        self.max_len = max_len

    # Dataset의 전체 샘플 개수를 반환합니다.
    def __len__(self):
        # 리뷰 리스트의 길이가 전체 데이터 개수입니다.
        return len(self.reviews)

    # 특정 index에 해당하는 하나의 샘플을 반환합니다.
    def __getitem__(self, index):
        # index 위치의 리뷰를 문자열로 변환하여 가져옵니다.
        review = str(self.reviews[index])

        # index 위치의 감성 레이블을 정수로 가져옵니다.
        sentiment = int(self.sentiments[index])

        # 최신 transformers에서는 encode_plus() 대신 tokenizer(...) 호출 방식을 사용하는 것이 안전합니다.
        # tokenizer(...)는 내부적으로 문장을 토큰화하고, 토큰 ID 변환, 패딩, 자르기, attention_mask 생성을 한 번에 수행합니다.
        encoded = self.tokenizer(
            review,                         # 토큰화할 원본 리뷰 문장입니다.
            add_special_tokens=True,         # [CLS], [SEP] 같은 특수 토큰을 자동으로 추가합니다.
            max_length=self.max_len,         # 문장의 최대 토큰 길이를 지정합니다.
            padding="max_length",           # 짧은 문장은 max_length까지 [PAD] 토큰으로 채웁니다.
            truncation=True,                 # 긴 문장은 max_length에 맞게 자릅니다.
            return_attention_mask=True,      # 실제 토큰과 패딩 토큰을 구분하는 attention_mask를 반환합니다.
            return_token_type_ids=False,     # 단일 문장 분류이므로 token_type_ids는 반환하지 않습니다.
            return_tensors="pt"              # 결과를 PyTorch Tensor 형태로 반환합니다.
        )

        # DataLoader가 사용할 딕셔너리 형태로 하나의 샘플을 반환합니다.
        return {
            "input_ids": encoded["input_ids"].squeeze(0),                # shape을 [1, max_len]에서 [max_len]으로 줄입니다.
            "attention_mask": encoded["attention_mask"].squeeze(0),      # attention_mask도 [max_len] 형태로 만듭니다.
            "labels": torch.tensor(sentiment, dtype=torch.long)           # 분류 정답은 long 타입 Tensor로 변환합니다.
        }

## 7. Tokenizer 다운로드와 Dataset 객체 생성

 Hugging Face에서 `kykim/bert-kor-base` Tokenizer를 다운로드하고, Train/Validation/Test Dataset 객체를 생성합니다.

In [12]:
# 사용할 한국어 BERT 모델 이름을 지정합니다.
bert_model_name = "kykim/bert-kor-base"

# Hugging Face Hub에서 사전 학습된 BERT Tokenizer를 다운로드합니다.
tokenizer = BertTokenizerFast.from_pretrained(bert_model_name)

# BERT에 입력할 최대 토큰 길이를 지정합니다.
MAX_LEN = 128

# Train DataFrame을 PyTorch Dataset 객체로 변환합니다.
train_dataset = BertMovieReviewDataset(
    reviews=train_set["review"].tolist(),          # Train 리뷰 문장을 리스트로 전달합니다.
    sentiments=train_set["sentiment"].tolist(),    # Train 정답 레이블을 리스트로 전달합니다.
    tokenizer=tokenizer,                            # 앞에서 다운로드한 Tokenizer를 전달합니다.
    max_len=MAX_LEN                                 # 최대 토큰 길이를 전달합니다.
)

# Validation DataFrame을 PyTorch Dataset 객체로 변환합니다.
valid_dataset = BertMovieReviewDataset(
    reviews=valid_set["review"].tolist(),          # Validation 리뷰 문장을 리스트로 전달합니다.
    sentiments=valid_set["sentiment"].tolist(),    # Validation 정답 레이블을 리스트로 전달합니다.
    tokenizer=tokenizer,                            # 같은 Tokenizer를 사용합니다.
    max_len=MAX_LEN                                 # 같은 최대 토큰 길이를 사용합니다.
)

# Test DataFrame을 PyTorch Dataset 객체로 변환합니다.
test_dataset = BertMovieReviewDataset(
    reviews=test_set["review"].tolist(),           # Test 리뷰 문장을 리스트로 전달합니다.
    sentiments=test_set["sentiment"].tolist(),     # Test 정답 레이블을 리스트로 전달합니다.
    tokenizer=tokenizer,                            # 같은 Tokenizer를 사용합니다.
    max_len=MAX_LEN                                 # 같은 최대 토큰 길이를 사용합니다.
)

# 첫 번째 Train 샘플의 Tensor 구조를 확인합니다.
print(train_dataset[0])

{'input_ids': tensor([    2, 15989, 14018,  8487, 14584, 35969,  8172,  8202, 39721, 16623,
            3,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0, 

## 8. DataLoader 생성

Dataset은 한 개의 샘플을 반환하고, DataLoader는 여러 샘플을 묶어 미니배치 단위로 모델에 공급합니다.

In [13]:
# GPU 메모리에 맞게 Batch Size를 설정합니다. 메모리 부족 시 8 또는 4로 줄입니다.
BATCH_SIZE = 16

# Train Dataset을 미니배치 단위로 섞어서 공급하는 DataLoader를 생성합니다.
train_loader = DataLoader(
    train_dataset,          # 학습용 Dataset입니다.
    batch_size=BATCH_SIZE,  # 한 번에 처리할 샘플 수입니다.
    shuffle=True            # 학습 시 데이터 순서를 섞어 일반화 성능을 높입니다.
)

# Validation Dataset을 미니배치 단위로 공급하는 DataLoader를 생성합니다.
valid_loader = DataLoader(
    valid_dataset,          # 검증용 Dataset입니다.
    batch_size=BATCH_SIZE,  # 검증에서도 같은 Batch Size를 사용합니다.
    shuffle=False           # 검증은 순서를 섞을 필요가 없습니다.
)

# Test Dataset을 미니배치 단위로 공급하는 DataLoader를 생성합니다.
test_loader = DataLoader(
    test_dataset,           # 평가용 Dataset입니다.
    batch_size=BATCH_SIZE,  # 평가용 Batch Size입니다.
    shuffle=False           # 평가도 순서를 섞지 않습니다.
)

## 9. Pre-trained BERT 모델 다운로드와 Fine-tuning 설정

사전 학습된 한국어 BERT 모델을 불러온 뒤, Fine-tuning 전략에 따라 일부 계층만 학습할 수 있도록 설정합니다.

In [14]:
# 이진 분류이므로 출력 클래스 개수를 2로 지정합니다.
NUM_LABELS = 2

# 문장 분류용 BERT 모델을 다운로드합니다.
model = BertForSequenceClassification.from_pretrained(
    bert_model_name,      # 사용할 사전 학습 모델 이름입니다.
    num_labels=NUM_LABELS # 출력 클래스 수입니다. 부정/긍정이므로 2입니다.
)

# 모델을 CPU 또는 GPU 장치로 이동합니다.
model = model.to(device)

# Fine-tuning 전략을 지정합니다. 0은 전체 학습, 1은 BERT 전체 고정, 2는 pooler만 학습, 3은 마지막 encoder와 pooler만 학습입니다.
tl_strategy = 3

# 전략 1: BERT 본체 전체를 고정하고 분류기만 학습합니다.
if tl_strategy == 1:
    # BERT 본체의 모든 파라미터를 순회합니다.
    for name, param in model.bert.named_parameters():
        # 현재 파라미터 이름을 확인용으로 출력합니다.
        print(name)

        # 해당 파라미터가 학습되지 않도록 gradient 계산을 끕니다.
        param.requires_grad = False

# 전략 2: pooler를 제외한 대부분의 BERT 본체를 고정합니다.
elif tl_strategy == 2:
    # BERT 본체의 모든 파라미터를 순회합니다.
    for name, param in model.bert.named_parameters():
        # 이름이 pooler로 시작하지 않는 파라미터만 고정합니다.
        if not name.startswith("pooler"):
            # 해당 파라미터가 학습되지 않도록 설정합니다.
            param.requires_grad = False

# 전략 3: 마지막 Encoder Layer와 pooler만 학습합니다.
elif tl_strategy == 3:
    # BERT Base 계열은 보통 encoder layer가 0~11까지 존재하므로 마지막 layer는 layer.11입니다.
    last_layer_name = "layer.11"

    # BERT 본체의 모든 파라미터를 순회합니다.
    for name, param in model.bert.named_parameters():
        # pooler도 아니고 마지막 encoder layer도 아니면 고정합니다.
        if (not name.startswith("pooler")) and (last_layer_name not in name):
            # 해당 파라미터가 학습되지 않도록 gradient 계산을 끕니다.
            param.requires_grad = False

# 학습 가능한 파라미터 수를 계산합니다.
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

# 전체 파라미터 수를 계산합니다.
total_params = sum(p.numel() for p in model.parameters())

# 학습 가능한 파라미터 비율을 출력합니다.
print(f"학습 가능 파라미터: {trainable_params:,} / 전체 파라미터: {total_params:,}")

config.json:   0%|          | 0.00/725 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/476M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: kykim/bert-kor-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were 

model.safetensors:   0%|          | 0.00/476M [00:00<?, ?B/s]

학습 가능 파라미터: 7,680,002 / 전체 파라미터: 118,298,882


## 10. 학습 하이퍼파라미터와 Optimizer 설정

Epoch, Batch Size, Weight Decay 등을 설정합니다. 여기서는 PyTorch 학습 루프를 직접 작성하기 위해 Optimizer와 Scheduler를 구성합니다.

In [15]:
# 전체 학습 반복 횟수를 지정합니다.
EPOCHS = 1

# AdamW Optimizer의 학습률을 지정합니다.
LEARNING_RATE = 2e-5

# Weight Decay는 과적합을 줄이기 위한 정규화 계수입니다.
WEIGHT_DECAY = 0.01

# Warmup Step은 학습 초반 학습률을 천천히 올리는 단계 수입니다.
WARMUP_STEPS = 0

# 학습 가능한 파라미터만 Optimizer에 전달합니다.
optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), # requires_grad=True인 파라미터만 업데이트합니다.
    lr=LEARNING_RATE,                                      # 학습률을 설정합니다.
    weight_decay=WEIGHT_DECAY                              # Weight Decay를 설정합니다.
)

# 전체 학습 Step 수를 계산합니다.
total_training_steps = len(train_loader) * EPOCHS

# 선형 학습률 스케줄러를 생성합니다.
scheduler = get_linear_schedule_with_warmup(
    optimizer,                              # 학습률을 조정할 Optimizer입니다.
    num_warmup_steps=WARMUP_STEPS,          # Warmup 단계 수입니다.
    num_training_steps=total_training_steps # 전체 학습 단계 수입니다.
)

## 11. 학습 함수와 평가 함수 정의

모델 학습은 `train_one_epoch`에서 수행하고, Validation/Test 평가는 `evaluate`에서 수행합니다.

In [16]:
# 한 Epoch 동안 모델을 학습하는 함수를 정의합니다.
def train_one_epoch(model, data_loader, optimizer, scheduler, device):
    # 모델을 학습 모드로 전환하여 Dropout 등이 활성화되게 합니다.
    model.train()

    # Epoch 전체 손실을 누적할 변수를 초기화합니다.
    total_loss = 0.0

    # 실제 정답 레이블을 저장할 리스트를 생성합니다.
    all_labels = []

    # 모델 예측 레이블을 저장할 리스트를 생성합니다.
    all_preds = []

    # DataLoader에서 미니배치를 하나씩 꺼내며 진행률을 표시합니다.
    for batch in tqdm(data_loader, desc="Training"):
        # input_ids Tensor를 학습 장치로 이동합니다.
        input_ids = batch["input_ids"].to(device)

        # attention_mask Tensor를 학습 장치로 이동합니다.
        attention_mask = batch["attention_mask"].to(device)

        # labels Tensor를 학습 장치로 이동합니다.
        labels = batch["labels"].to(device)

        # 이전 Step에서 계산된 gradient를 초기화합니다.
        optimizer.zero_grad()

        # BERT 모델에 입력을 넣어 loss와 logits를 계산합니다.
        outputs = model(
            input_ids=input_ids,             # 토큰 ID 입력입니다.
            attention_mask=attention_mask,   # 패딩 위치를 무시하기 위한 마스크입니다.
            labels=labels                    # 정답 레이블을 넣으면 loss가 자동 계산됩니다.
        )

        # 모델이 계산한 CrossEntropyLoss를 가져옵니다.
        loss = outputs.loss

        # loss를 기준으로 역전파를 수행하여 gradient를 계산합니다.
        loss.backward()

        # gradient 폭주를 방지하기 위해 gradient norm을 제한합니다.
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Optimizer가 파라미터를 업데이트합니다.
        optimizer.step()

        # Scheduler가 학습률을 한 Step 갱신합니다.
        scheduler.step()

        # 현재 배치의 loss 값을 누적합니다.
        total_loss += loss.item()

        # logits에서 가장 큰 값을 가진 클래스 인덱스를 예측값으로 선택합니다.
        preds = torch.argmax(outputs.logits, dim=1)

        # 정답 레이블을 CPU 리스트로 변환하여 누적합니다.
        all_labels.extend(labels.detach().cpu().numpy())

        # 예측 레이블을 CPU 리스트로 변환하여 누적합니다.
        all_preds.extend(preds.detach().cpu().numpy())

    # 평균 loss를 계산합니다.
    avg_loss = total_loss / len(data_loader)

    # 정확도를 계산합니다.
    accuracy = accuracy_score(all_labels, all_preds)

    # 평균 loss와 정확도를 반환합니다.
    return avg_loss, accuracy

# 모델을 평가하는 함수를 정의합니다.
def evaluate(model, data_loader, device):
    # 모델을 평가 모드로 전환하여 Dropout 등을 비활성화합니다.
    model.eval()

    # 평가 전체 손실을 누적할 변수를 초기화합니다.
    total_loss = 0.0

    # 실제 정답 레이블을 저장할 리스트를 생성합니다.
    all_labels = []

    # 모델 예측 레이블을 저장할 리스트를 생성합니다.
    all_preds = []

    # 평가 중에는 gradient 계산이 필요 없으므로 비활성화합니다.
    with torch.no_grad():
        # DataLoader에서 미니배치를 하나씩 꺼냅니다.
        for batch in tqdm(data_loader, desc="Evaluating"):
            # input_ids Tensor를 평가 장치로 이동합니다.
            input_ids = batch["input_ids"].to(device)

            # attention_mask Tensor를 평가 장치로 이동합니다.
            attention_mask = batch["attention_mask"].to(device)

            # labels Tensor를 평가 장치로 이동합니다.
            labels = batch["labels"].to(device)

            # 모델에 입력을 넣어 loss와 logits를 계산합니다.
            outputs = model(
                input_ids=input_ids,           # 토큰 ID 입력입니다.
                attention_mask=attention_mask, # 패딩 위치를 알려주는 마스크입니다.
                labels=labels                  # 정답 레이블입니다.
            )

            # 현재 배치의 loss를 누적합니다.
            total_loss += outputs.loss.item()

            # logits에서 가장 큰 값을 가진 클래스를 예측값으로 선택합니다.
            preds = torch.argmax(outputs.logits, dim=1)

            # 정답 레이블을 CPU 리스트로 변환하여 누적합니다.
            all_labels.extend(labels.detach().cpu().numpy())

            # 예측 레이블을 CPU 리스트로 변환하여 누적합니다.
            all_preds.extend(preds.detach().cpu().numpy())

    # 평균 loss를 계산합니다.
    avg_loss = total_loss / len(data_loader)

    # 정확도를 계산합니다.
    accuracy = accuracy_score(all_labels, all_preds)

    # 평균 loss, 정확도, 전체 정답, 전체 예측을 반환합니다.
    return avg_loss, accuracy, all_labels, all_preds

## 12. 모델 학습

Validation 성능을 확인하면서 모델을 학습합니다. 데이터가 크면 시간이 오래 걸릴 수 있으므로 GPU 사용을 권장합니다.

In [17]:
# Epoch별 학습을 반복합니다.
for epoch in range(EPOCHS):
    # 현재 Epoch 번호를 출력합니다.
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    # Train 데이터로 한 Epoch 학습을 수행합니다.
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scheduler, device)

    # Validation 데이터로 모델 성능을 평가합니다.
    valid_loss, valid_acc, _, _ = evaluate(model, valid_loader, device)

    # Train 손실과 정확도를 출력합니다.
    print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_acc:.4f}")

    # Validation 손실과 정확도를 출력합니다.
    print(f"Valid Loss: {valid_loss:.4f} | Valid Accuracy: {valid_acc:.4f}")


Epoch 1/1


Training:   0%|          | 0/5250 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/2250 [00:00<?, ?it/s]

Train Loss: 0.3986 | Train Accuracy: 0.8207
Valid Loss: 0.3422 | Valid Accuracy: 0.8576


## 13. Test 데이터 평가와 예측 결과 확인

 Test Dataset을 사용하여 최종 예측을 수행합니다.

In [18]:
# Test 데이터로 최종 평가를 수행합니다.
test_loss, test_acc, test_labels, test_preds = evaluate(model, test_loader, device)

# Test 손실을 출력합니다.
print(f"Test Loss: {test_loss:.4f}")

# Test 정확도를 출력합니다.
print(f"Test Accuracy: {test_acc:.4f}")

# 부정/긍정 클래스별 정밀도, 재현율, F1-score를 출력합니다.
print(classification_report(
    test_labels,                 # 실제 정답 레이블입니다.
    test_preds,                  # 모델 예측 레이블입니다.
    target_names=["부정", "긍정"] # 클래스 이름입니다.
))

Evaluating:   0%|          | 0/1875 [00:00<?, ?it/s]

Test Loss: 0.3448
Test Accuracy: 0.8565
              precision    recall  f1-score   support

          부정       0.87      0.84      0.85     15034
          긍정       0.85      0.87      0.86     14965

    accuracy                           0.86     29999
   macro avg       0.86      0.86      0.86     29999
weighted avg       0.86      0.86      0.86     29999



## 14. 새 문장 감성 예측 함수

학습된 BERT 모델을 사용하여 사용자가 입력한 새 영화 리뷰가 긍정인지 부정인지 예측합니다.

In [20]:
# 새 리뷰 문장 하나를 입력받아 감성을 예측하는 함수를 정의합니다.
def predict_sentiment(text, model, tokenizer, device, max_len=128):
    # 모델을 평가 모드로 전환합니다.
    model.eval()

    # 입력 문장을 BERT 입력 형식으로 변환합니다.
    encoded = tokenizer(
        text,                          # 예측할 원본 문장입니다.
        add_special_tokens=True,        # [CLS], [SEP] 토큰을 추가합니다.
        max_length=max_len,             # 최대 토큰 길이를 지정합니다.
        padding="max_length",          # 짧은 문장은 패딩합니다.
        truncation=True,                # 긴 문장은 자릅니다.
        return_attention_mask=True,     # attention_mask를 반환합니다.
        return_tensors="pt"             # PyTorch Tensor로 반환합니다.
    )

    # input_ids를 학습 장치로 이동합니다.
    input_ids = encoded["input_ids"].to(device)

    # attention_mask를 학습 장치로 이동합니다.
    attention_mask = encoded["attention_mask"].to(device)

    # 예측 과정에서는 gradient가 필요 없으므로 비활성화합니다.
    with torch.no_grad():
        # 모델에 입력을 넣어 logits를 얻습니다.
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        # logits를 softmax에 통과시켜 클래스별 확률로 변환합니다.
        probabilities = torch.softmax(outputs.logits, dim=1)

        # 가장 확률이 높은 클래스 인덱스를 선택합니다.
        predicted_class = torch.argmax(probabilities, dim=1).item()

    # 클래스 인덱스를 사람이 읽기 쉬운 감성명으로 변환합니다.
    label_name = "긍정" if predicted_class == 1 else "부정"

    # 예측 감성명과 클래스별 확률을 반환합니다.
    return label_name, probabilities.squeeze(0).detach().cpu().numpy()

# 예측에 사용할 예시 리뷰 문장을 지정합니다.
example_review = "배우들의 연기가 좋고 스토리도 재미있었습니다."

# 예시 리뷰의 감성을 예측합니다.
label, probs = predict_sentiment(example_review, model, tokenizer, device, MAX_LEN)

# 예측 결과를 출력합니다.
print("입력 문장:", example_review)

# 예측 감성명을 출력합니다.
print("예측 감성:", label)

# 부정/긍정 확률을 출력합니다.
print("[부정 확률, 긍정 확률]:", probs)

입력 문장: 배우들의 연기가 좋고 스토리도 재미있었습니다.
예측 감성: 긍정
[부정 확률, 긍정 확률]: [0.00531116 0.99468887]
